## Preprocessing of Home Loan Dataset

This notebook implements preprocessing steps based on the comprehensive EDA findings and recommendations. 

We'll follow the evidence-based approach from the EDA report to ensure our preprocessing aligns with the data patterns discovered.
Based on the EDA report, we will:

1. **Handle Skewed Variables** - Log-transform `ApplicantIncome`, `CoApllicantIncome`, `Loan Amount`, `credit History`
2. **Outlier Treatment** - IQR-capping for extreme - `ApplicantIncome`, `CoApllicantIncome`, `Loan Amount` 
3. **Feature Engineering** - Create `total income `, `loan_income ratio`, `income_term`
4. **Feature Selection** - Keep high-signal features, evaluate low-signal ones
5. **Scaling** - StandardScaler for distance-based models
6. **Target Handling** - Classification approach with stratified splits

**Key EDA Evidence to Implement**

- **High-signal features**: `Credit_History`
- **Low-signal features**: `CoapplicantIncome`, `LoanAmount`, `Loan_Amount_Term` `ApplicantIncome` (evaluate for removal)
- **Skewed variables**: `ApplicantIncome`, `CoApllicantIncome`, `Loan Amount` (log-transform)
- **Feature engineering**: Create `total_inocme` and `loan_income ratio`, `income_term`

In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [62]:
# Preprocessing libraries
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

In [63]:
# Statistical libraries
from scipy import stats
from scipy.stats import zscore, skew

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

Libraries imported successfully!


In [64]:
df = pd.read_csv("cleaned_homeloan_dataset.csv")
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,0
1,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,1
2,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,1
3,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,1
4,LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,1


In [65]:
# create a copy for preprocessing

df_processed = df.copy()

In [ ]:


# # Drop the 'Loan_ID' column from df_processed
df_processed = df_processed.drop(columns="Loan_ID")

# Display the first few rows
df_processed.head()


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,0
1,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,1
2,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,1
3,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,1
4,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,1


#### **2. EDA-Based Data Quality Assessment**

In [67]:
# 1. Check for missing values (EDA showed no missing values)
print("\n1. Missing Values:")
missing_values = df_processed.isnull().sum()
if missing_values.sum() > 0:
    print(missing_values[missing_values > 0])
else:
    print("No missing values found (as expected from EDA)")

# 2. Check for duplicates
print("\n2. Duplicate Rows:")
duplicates = df_processed.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    print(f"Percentage of duplicates: {(duplicates/len(df_processed))*100:.2f}%")

# 3. Check skewness for variables identified in EDA as right-skewed
print("\n3. Skewness Analysis (EDA identified right-skewed variables):")
skewed_vars = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
for var in skewed_vars:
    if var in df_processed.columns:
        skewness = skew(df_processed[var])
        print(f"{var}: skewness = {skewness:.3f} ({'right-skewed' if skewness > 0.5 else 'approximately normal'})")

# 4. Check correlation with target (EDA evidence)
print("\n4. Correlation with Quality (EDA Evidence):")
# correlations = df_processed.corr()['Loan_Status'].sort_values(key=abs, ascending=False)
correlations = df_processed.select_dtypes(include=['number']).corr()['Loan_Status'].sort_values(key=abs, ascending=False)

print("High-signal features (|correlation| > 0.2):")
high_signal = correlations[abs(correlations) > 0.2].drop('Loan_Status')
for feature, corr in high_signal.items():
    print(f"  {feature}: {corr:.3f}")

print("\nLow-signal features (|correlation| < 0.1):")
low_signal = correlations[abs(correlations) < 0.1]
for feature, corr in low_signal.items():

    print(f"  {feature}: {corr:.3f}")


1. Missing Values:
No missing values found (as expected from EDA)

2. Duplicate Rows:
Number of duplicate rows: 0

3. Skewness Analysis (EDA identified right-skewed variables):
ApplicantIncome: skewness = 6.558 (right-skewed)
CoapplicantIncome: skewness = 7.411 (right-skewed)
LoanAmount: skewness = 2.671 (right-skewed)

4. Correlation with Quality (EDA Evidence):
High-signal features (|correlation| > 0.2):
  Credit_History: 0.430

Low-signal features (|correlation| < 0.1):
  CoapplicantIncome: -0.065
  LoanAmount: -0.037
  Loan_Amount_Term: -0.031
  Dependents: 0.015
  ApplicantIncome: -0.005


#### **3. Log-Transform Skewed Variables (EDA Recommendation)**

Based on our EDA findings, we will transforming the right-skewed variables

In [68]:
# Log-transform skewed variables as recommended by EDA
print("=== LOG-TRANSFORMING SKEWED VARIABLES ===")
print("EDA identified these variables as right-skewed and recommended log transformation:")

# Variables to log-transform based on EDA findings
skewed_vars = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']

for var in skewed_vars:
    if var in df_processed.columns:
        # Check if variable has zero or negative values
        min_val = df_processed[var].min()
        if min_val <= 0:
            # Use log1p for variables with zeros
            df_processed[f'{var}_log'] = np.log1p(df_processed[var])
            print(f"✓ {var}: Applied log1p transformation (had {min_val:.3f} minimum value)")
        else:
            # Use log for positive values only
            df_processed[f'{var}_log'] = np.log(df_processed[var])
            print(f"✓ {var}: Applied log transformation")
        
        # Check skewness before and after
        original_skew = skew(df_processed[var])
        transformed_skew = skew(df_processed[f'{var}_log'])
        print(f"  Original skewness: {original_skew:.3f} → Transformed skewness: {transformed_skew:.3f}")

print(f"\nDataset shape after log transformation: {df_processed.shape}")
print("New log-transformed columns:", [col for col in df_processed.columns if '_log' in col])

=== LOG-TRANSFORMING SKEWED VARIABLES ===
EDA identified these variables as right-skewed and recommended log transformation:
✓ ApplicantIncome: Applied log transformation
  Original skewness: 6.558 → Transformed skewness: 0.469
✓ CoapplicantIncome: Applied log1p transformation (had 0.000 minimum value)
  Original skewness: 7.411 → Transformed skewness: -0.186
✓ LoanAmount: Applied log transformation
  Original skewness: 2.671 → Transformed skewness: -0.193

Dataset shape after log transformation: (592, 15)
New log-transformed columns: ['ApplicantIncome_log', 'CoapplicantIncome_log', 'LoanAmount_log']


#### **4. Outlier Treatment (EDA Recommendation)**

ased on EDA findings, handle outliers using IQR-capping metho

In [69]:
# Outlier treatment based on EDA recommendations
print("=== OUTLIER TREATMENT (IQR-CAPPING METHOD) ===")
print("EDA recommended IQR-capping for extreme acidity/sulphates to preserve data points")

# Define numerical columns (excluding target)
numerical_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
if 'quality' in numerical_cols:
    numerical_cols.remove('quality')

print(f"Treating outliers in {len(numerical_cols)} numerical features...")

# Apply IQR-capping method
outliers_capped = 0
for col in numerical_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count outliers before capping
    outliers_before = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()
    
    if outliers_before > 0:
        # Cap outliers
        df_processed[col] = np.where(df_processed[col] < lower_bound, lower_bound, df_processed[col])
        df_processed[col] = np.where(df_processed[col] > upper_bound, upper_bound, df_processed[col])
        outliers_capped += outliers_before
        print(f"✓ {col}: Capped {outliers_before} outliers")

print(f"\nTotal outliers capped: {outliers_capped}")
print(f"Dataset shape after outlier treatment: {df_processed.shape}")


=== OUTLIER TREATMENT (IQR-CAPPING METHOD) ===
EDA recommended IQR-capping for extreme acidity/sulphates to preserve data points
Treating outliers in 10 numerical features...
✓ Dependents: Capped 49 outliers
✓ ApplicantIncome: Capped 49 outliers
✓ CoapplicantIncome: Capped 18 outliers
✓ LoanAmount: Capped 39 outliers
✓ Loan_Amount_Term: Capped 85 outliers
✓ Credit_History: Capped 134 outliers
✓ ApplicantIncome_log: Capped 30 outliers
✓ LoanAmount_log: Capped 34 outliers

Total outliers capped: 438
Dataset shape after outlier treatment: (592, 15)


In [70]:
df_processed.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,ApplicantIncome_log,CoapplicantIncome_log,LoanAmount_log
0,Male,Yes,1.0,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,0,8.430109,7.319202,4.852030
1,Male,Yes,0.0,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban,1,8.006368,0.000000,4.189655
2,Male,Yes,0.0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,1,7.856707,7.765993,4.787492
3,Male,No,0.0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,1,8.699515,0.000000,4.948760
4,Male,Yes,2.0,Graduate,Yes,5417.0,4196.0,267.0,360.0,1.0,Urban,1,8.597297,8.342125,5.587249


#### **6. Feature Engineering**

Implement the specific feature engineering recommendations from the EDA report

In [ ]:
# Creating Total Income
df_processed["Total_income"] = df_processed["ApplicantIncome"] + df_processed["CoapplicantIncome"]
print("Total amount of income is",df_processed["Total_income"].head(2))


# Creating Loan Income Ratio
df_processed["Income_ratio"] = np.divide(df_processed["Total_income"],df_processed["LoanAmount"])
print("Income ratio is",df_processed["Total_income"].head(2))


#Creating Income_per_term
df_processed["Income_per_term"] = np.divide(df_processed["Total_income"], df_processed["Loan_Amount_Term"])
print("Income_per_term",df_processed["Total_income"].head(2))

Total amount of income is 0    6091.0
1    3000.0
Name: Total_income, dtype: float64
Income ratio is 0    6091.0
1    3000.0
Name: Total_income, dtype: float64
Income_per_term 0    6091.0
1    3000.0
Name: Total_income, dtype: float64


In [79]:
df_processed.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,ApplicantIncome_log,CoapplicantIncome_log,LoanAmount_log,Total_income,Income_ratio,Income_per_term
0,Male,Yes,1.0,Graduate,No,4583.0,1508.0,128.0,360.0,1.0,Rural,0,8.430109,7.319202,4.852030,6091.0,47.585938,16.919444
1,Male,Yes,0.0,Graduate,Yes,3000.0,0.0,66.0,360.0,1.0,Urban,1,8.006368,0.000000,4.189655,3000.0,45.454545,8.333333
2,Male,Yes,0.0,Not Graduate,No,2583.0,2358.0,120.0,360.0,1.0,Urban,1,7.856707,7.765993,4.787492,4941.0,41.175000,13.725000
3,Male,No,0.0,Graduate,No,6000.0,0.0,141.0,360.0,1.0,Urban,1,8.699515,0.000000,4.948760,6000.0,42.553191,16.666667
4,Male,Yes,2.0,Graduate,Yes,5417.0,4196.0,267.0,360.0,1.0,Urban,1,8.597297,8.342125,5.587249,9613.0,36.003745,26.702778
